# INVEN 공식 문서(GUIDE) 데이터 전처리

> - 데이터 파일 : RAG/maple_guides.json


In [1]:
# [환경 설정] 필요한 라이브러리와 한글 폰트 설정

import re
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kiwipiepy import Kiwi
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

kiwi = Kiwi()   # 한국어 형태소 분석기(자바 불필요)

In [45]:
# INVEN 공식문서(GUIDE) 가져오기
with open('../data/RAG/maple_guides.json', encoding='utf-8') as f :
    guide_dict = json.load(f)

# JSON 파일의 DICT -> 판다스 DataFrame 으로 변환
guide_df = pd.DataFrame(guide_dict)

# GUIDE 데이터 확인
display(guide_df.info())
display(guide_df['category'].unique())
display(guide_df.head())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   article_id  106 non-null    int64 
 1   title       106 non-null    object
 2   content     106 non-null    object
 3   url         106 non-null    object
 4   category    106 non-null    object
 5   board_id    106 non-null    int64 
dtypes: int64(2), object(4)
memory usage: 5.1+ KB


None

array(['기초 가이드', '성장', '아이템', '사냥/보스 컨텐츠', '스페셜 컨텐츠', '커뮤니티', '거래',
       '캐시 & 코디', '기타/TIP'], dtype=object)

,article_id,title,content,url,category,board_id
0,272,게임 시작,■ 목차\n1. 메이플스토리 계정 만들기\n2. 게임 설치하기\n\n\n\n\n\n...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337
1,373,캐릭터 생성/삭제,■ 목차\n1. 캐릭터 직업 선택하기\n2. 캐릭터 설정하기\n3. 캐릭터 삭제하기...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337
2,307,캐릭터 이름 변경,■ 목차\n1. 캐릭터 이름 변경 방법\n2. 캐릭터 이름 변경 시 유의사항\n\n...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337
3,417,캐릭터 프리셋,■ 목차\n\n1. 캐릭터 프리셋이란?\n\n2. 캐릭터 프리셋 사용 방법\n\n3...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337
4,432,조작법/키세팅,■ 목차\n1. 기본 조작키\n2. 단축키 설정하기\n3. 퀵슬롯 설정하기\n4. ...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337


In [ ]:
# 3단계 정제·정규화 프로세스 (정규표현식으로 노이즈 제거)

def process(text) :
    # 1) 특수문자·이모지 제거 — 한글·영문·숫자·공백이 '아닌' 글자를 공백으로
    # [^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]
    # [^] 대괄호 내에서 꺽쇠 ^ 아닌것들을 의미
    # [^abc] : a,b,c가 아닌 문자 1개와 매칭
    # [^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s] ^는 제외라는 의미 (not의 의미)
    step1 = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', text)
    # 반복된 줄바꿈(개행) 제거
    step1_1 = re.sub(r'\n+', '\n', step1)
    # 2) 반복문자 축약 — 같은 글자가 3번 이상이면 2번으로 (ㅋㅋㅋㅋ → ㅋㅋ)
    step2 = re.sub(r'(.)\1{2,}', r'\1\1', step1_1)
    # (.)\1{2,}
    # . : 글자 아무거나 1개 매칭
    # (.) : 글자 1개 매칭된 것을 캡쳐 (변수화)
    # (.)\1 : \1-첫번째 캡쳐된 글자를 의미
    # (.)\1{2,} : 같은 글자가 3번이상 반복하는 것과 매칭
    # (.).\1 : 3글자 회문, ex) 기러기, 토마토
    # 3) 공백 정규화 — 여러 칸을 한 칸으로, 양끝 공백 제거
    result = re.sub(r'\s+', ' ', step2).strip()
    return result

In [49]:
# guide_df['content'] 데이터의 정제·정규화 후 guide_narm 데이터프레임 및 'normalized' 칼럼 생성

# 데이터프레임 복제
guide_norm = guide_df.copy()

guide_norm['normalized_content'] = guide_df['content'].apply(process)

display(guide_norm.head())

,article_id,title,content,url,category,board_id,normalized_content
0,272,게임 시작,■ 목차\n1. 메이플스토리 계정 만들기\n2. 게임 설치하기\n\n\n\n\n\n...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 메이플스토리 계정 만들기 2 게임 설치하기 메이플스토리 계정 만들기 1 회...
1,373,캐릭터 생성/삭제,■ 목차\n1. 캐릭터 직업 선택하기\n2. 캐릭터 설정하기\n3. 캐릭터 삭제하기...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 캐릭터 직업 선택하기 2 캐릭터 설정하기 3 캐릭터 삭제하기 캐릭터 생성 ...
2,307,캐릭터 이름 변경,■ 목차\n1. 캐릭터 이름 변경 방법\n2. 캐릭터 이름 변경 시 유의사항\n\n...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 캐릭터 이름 변경 방법 2 캐릭터 이름 변경 시 유의사항 캐릭터 이름 변경...
3,417,캐릭터 프리셋,■ 목차\n\n1. 캐릭터 프리셋이란?\n\n2. 캐릭터 프리셋 사용 방법\n\n3...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 캐릭터 프리셋이란 2 캐릭터 프리셋 사용 방법 3 장비 프리셋 사용 방법 ...
4,432,조작법/키세팅,■ 목차\n1. 기본 조작키\n2. 단축키 설정하기\n3. 퀵슬롯 설정하기\n4. ...,https://maplestory.nexon.com/Guide/N23GameInfo...,기초 가이드,429467337,목차 1 기본 조작키 2 단축키 설정하기 3 퀵슬롯 설정하기 4 스킬 매크로 등록하...


In [51]:
# 데이터프레임 -> JSON 변환 후 파일 저장

guide_norm.to_json('../data/RAG/maple_guide_normalized.json', orient='records', force_ascii=False, indent=4)
